<img src="../images/cads-logo.png" style="height: 100px;" align=left> 
<img src="../images/sklearn-logo.png" style="height: 100px;" align=right>

# Supervised Machine Learning

# Table of Contents

- [Thinking about Model Validation](#Thinking-about-Model-Validation)
- [Cross Validation](#Cross-Validation)
- [Model validation the wrong way](#Model-validation-the-wrong-way)
    - [Question: Can you guess the result of the following cell?](#Question:-Can-you-guess-the-result-of-the-following-cell?)
- [Model validation the right way: Holdout sets](#Model-validation-the-right-way:-Holdout-sets)
- [Model validation via cross-validation](#Model-validation-via-cross-validation)
- [Grid Search](#Grid-Search)

# Thinking about Model Validation

In principle, model validation is very simple: after choosing a model and its hyperparameters, we can estimate how effective it is by applying it to some of the training data and comparing the prediction to the known value.

The following sections first show a naive approach to model validation and why it
fails, before exploring the use of holdout sets and cross-validation for more robust
model evaluation.

# Cross Validation

## Model validation the wrong way

Let's demonstrate the naive approach to validation using the Iris data, which we saw in the previous section. We will start by loading the data:

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()

In [2]:
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (150, 4)
Shape of y: (150,)


Next we choose a model and hyperparameters. Here we'll use a *k*-neighbors classifier with ``n_neighbors=1``.
This is a very simple and intuitive model that says "the label of an unknown point is the same as the label of its closest training point:"

<img src='../images/KNN.png'>

In [3]:
from sklearn.neighbors import KNeighborsClassifier
model= KNeighborsClassifier(n_neighbors=1)

Then we train the model, and use it to predict labels for data we already know:

In [4]:
model.fit(X, y)
y_model = model.predict(X)

Finally, we compute the fraction of correctly labeled points:

### Question: Can you guess the result of the following cell?

In [5]:
from sklearn.metrics import accuracy_score
accuracy_score(y, y_model)

1.0

We see an accuracy score of 1.0, which indicates that 100% of points were correctly labeled by our model!
But is this truly measuring the expected accuracy? Have we really come upon a model that we expect to be correct 100% of the time?

As you may have gathered, the answer is no.
In fact, this approach contains a fundamental flaw: *it trains and evaluates the model on the same data*.
Furthermore, the nearest neighbor model is an *instance-based* estimator that simply stores the training data, and predicts labels by comparing new data to these stored points: except in contrived cases, it will get 100% accuracy *every time!*

## Model validation the right way: Holdout sets

So what can be done?
A better sense of a model's performance can be found using what's known as a *holdout set*: that is, we hold back some subset of the data from the training of the model, and then use this holdout set to check the model performance.
This splitting can be done using the ``train_test_split`` utility in Scikit-Learn:

In [6]:
print(X.shape)
print(y.shape)

(150, 4)
(150,)


In [7]:
from sklearn.model_selection import train_test_split

# split the data with 50% in each set
X_train, X_test, y_train, y_test = train_test_split(X, y,random_state=0,train_size=0.5)

#fit the model
model.fit(X_train, y_train)

# fit and evaluate the model on the second set of data
y_model = model.predict(X_test)

accuracy_score(y_test, y_model)

0.9066666666666666

In [8]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(75, 4)
(75,)
(75, 4)
(75,)


We see here a more reasonable result: the nearest-neighbor classifier is about 90% accurate on this hold-out set.
The hold-out set is similar to unknown data, because the model has not "seen" it before.

## Model validation via cross-validation

One disadvantage of using a holdout set for model validation is that we have lost a portion of our data to the model training.
In the above case, half the dataset does not contribute to the training of the model!
This is not optimal, and can cause problems – especially if the initial set of training data is small.

One way to address this is to use *cross-validation*; that is, to do a sequence of fits where each subset of the data is used both as a training set and as a validation set. Visually, it might look something like this:
<img src= "../images/2-fold-CV.png" style="height: 500px;">


Here we do two validation trials, alternately using each half of the data as a holdout set.
Using the split data from before, we could implement it like this:

**Question:**
Write the code that implements the accuracy described on the previous image

In [9]:
# solution
y1_model = model.fit(X_train, y_train).predict(X_test)
y2_model = model.fit(X_test, y_test).predict(X_train)
accuracy_score(y_test, y1_model), accuracy_score(y_train, y2_model)

(0.9066666666666666, 0.96)

What comes out are two accuracy scores, which we could combine (by, say, taking the mean) to get a better measure of the global model performance.
This particular form of cross-validation is a *two-fold cross-validation*—that is, one in which we have split the data into two sets and used each in turn as a validation set.

We could expand on this idea to use even more trials, and more folds in the data—for example, here is a visual depiction of five-fold cross-validation:

<img src='../images/CV.png'/>

Here we split the data into five groups, and use each of them in turn to evaluate the model fit on the other 4/5 of the data.
This would be rather tedious to do by hand, and so we can use Scikit-Learn's ``cross_val_score`` convenience routine to do it succinctly:

In [10]:
from sklearn.model_selection import cross_val_score

model= KNeighborsClassifier(n_neighbors=3)
scores = cross_val_score(model, X, y, cv=2)  
scores

array([0.96, 0.92])

In [11]:
print("Accuracy: {}".format(scores.mean()))

Accuracy: 0.94


By default, the score computed at each cv iteration is the `score` method of the estimator. It is possible to change this by using the scoring parameter. Take a look at all possible values for [scoring parameter](https://scikit-learn.org/stable/modules/model_evaluation.html).


In [12]:
scores_f1 = cross_val_score(model, X, y, cv=2, scoring='f1_macro')   
scores_f1, np.mean(scores_f1)

(array([0.95998399, 0.91987179]), np.float64(0.9399278942346169))

Repeating the validation across different subsets of the data gives us an even better idea of the performance of the algorithm.

Scikit-Learn implements a number of useful cross-validation schemes that are useful in particular situations; these are implemented via iterators in the ``cross_validation`` module.
For example, we might wish to go to the extreme case in which our number of folds is equal to the number of data points: that is, we train on all points but one in each trial.
This type of cross-validation is known as *leave-one-out* cross validation, and can be used as follows:

In [13]:
from sklearn.model_selection import LeaveOneOut

model= KNeighborsClassifier(n_neighbors=3)
scores = cross_val_score(model, X, y, cv=LeaveOneOut())
scores

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

Because we have 150 samples, the leave one out cross-validation yields scores for 150 trials, and the score indicates either successful (1.0) or unsuccessful (0.0) prediction.
Taking the mean of these gives an estimate of the error rate:

In [ ]:
scores.mean()

Other cross-validation schemes can be used similarly.
For a description of what is available in Scikit-Learn, use IPython to explore the ``sklearn.cross_validation`` submodule, or take a look at Scikit-Learn's online [cross-validation documentation](http://scikit-learn.org/stable/modules/cross_validation.html).

**Exercise:**
Try to classify Iris data using KNN for n_neighbors=4. Use 5-fold cross validation and use Accuracy, Precision, Recall, F1-score as evaluation metrics.

In [17]:
knn_5= KNeighborsClassifier(n_neighbors=4)

A_scores=cross_val_score(knn_5, X, y, cv=5)
A_mean=np.mean(A_scores)
print("Mean Accuracy: {}".format(A_mean))

P_scores=cross_val_score(knn_5, X, y, cv=5, scoring='precision_macro')
P_mean=np.mean(P_scores)
print("Mean Precision: {}".format(P_mean))

R_scores=cross_val_score(knn_5, X, y, cv=5, scoring='recall_macro')
R_mean=np.mean(R_scores)
print("Mean Recall: {}".format(R_mean))

F1_scores=cross_val_score(knn_5, X, y, cv=5, scoring='f1_macro')
F1_mean=np.mean(F1_scores)
print("Mean F1: {}".format(F1_mean))

Mean Accuracy: 0.9733333333333334
Mean Precision: 0.9757575757575758
Mean Recall: 0.9733333333333334
Mean F1: 0.9732664995822891


### Exercise
Try to classify 'indian_liver_patient.csv' data using KNN for n_neighbors=5 and Logistic Regression. Use 3-fold cross validation.

This data set contains 416 liver patient records and 167 non liver patient records collected from North East of Andhra Pradesh, India. The "Dataset" column is a class label used to divide groups into liver patient (liver disease) or not (no disease). 

In [18]:
liver = pd.read_csv("../Data/indian_liver_patient.csv")
liver.head(2)

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,0
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,0


In [19]:
liver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         583 non-null    int64  
 1   Gender                      583 non-null    object 
 2   Total_Bilirubin             583 non-null    float64
 3   Direct_Bilirubin            583 non-null    float64
 4   Alkaline_Phosphotase        583 non-null    int64  
 5   Alamine_Aminotransferase    583 non-null    int64  
 6   Aspartate_Aminotransferase  583 non-null    int64  
 7   Total_Protiens              583 non-null    float64
 8   Albumin                     583 non-null    float64
 9   Albumin_and_Globulin_Ratio  583 non-null    float64
 10  Dataset                     583 non-null    int64  
dtypes: float64(5), int64(5), object(1)
memory usage: 50.2+ KB


In [20]:
#Assuming 0=no disease and 1=disease
liver.Dataset.value_counts()

Dataset
0    416
1    167
Name: count, dtype: int64

In [21]:
X = liver.drop('Dataset', axis=1)
y = liver.Dataset

In [22]:
X_dum=pd.get_dummies(X, drop_first=True)
X_dum

,Age,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Gender_Male
0,65,0.7,0.1,187,16,18,6.8,3.3,0.90,False
1,62,10.9,5.5,699,64,100,7.5,3.2,0.74,True
2,62,7.3,4.1,490,60,68,7.0,3.3,0.89,True
3,58,1.0,0.4,182,14,20,6.8,3.4,1.00,True
4,72,3.9,2.0,195,27,59,7.3,2.4,0.40,True
...,...,...,...,...,...,...,...,...,...,...
578,60,0.5,0.1,500,20,34,5.9,1.6,0.37,True
579,40,0.6,0.1,98,35,31,6.0,3.2,1.10,True
580,52,0.8,0.2,245,48,49,6.4,3.2,1.00,True
581,31,1.3,0.5,184,29,32,6.8,3.4,1.00,True


In [23]:
### KNN

from sklearn.preprocessing import MinMaxScaler
x_scaled = MinMaxScaler().fit_transform(X_dum)


knn_5= KNeighborsClassifier(n_neighbors=5)

A_scores=cross_val_score(knn_5, x_scaled, y, cv=3, scoring='precision_macro')
A_mean=np.mean(A_scores)
print('Mean of precision: {}'.format(A_mean))

Mean of precision: 0.5797898665311592


In [24]:
### Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_5= LogisticRegression()

A_scores=cross_val_score(lr_5, x_scaled, y, cv=3, scoring='precision_macro')
A_mean=np.mean(A_scores)
print('Mean of precision: {}'.format(A_mean))

Mean of precision: 0.7758208345692875


# Grid Search
Grid search is the process of performing hyper parameter tuning in order to determine the optimal values for a given model. Scikit-Learn provides automated tools to do this in the grid search module.

Here is an example of using grid search to find the optimal KNN model. This can be set up using Scikit-Learn's ``GridSearchCV`` meta-estimator:

Now let's load breast_cancer data that is available from sklearn datasets.

In [25]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X = cancer.data
y= cancer.target
X.shape

(569, 30)

In [26]:
np.sum(y)

np.int64(357)

In [27]:
X_train, X_test, y_train, y_test=train_test_split(X,y,random_state=42, train_size=.80,stratify=y)

In [28]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_neighbors': np.arange(1, 20),
              'p': [1,2],
              'weights': ['uniform','distance']}

grid = GridSearchCV(KNeighborsClassifier(), 
                    param_grid, scoring='precision_macro',
                    cv=3)

In [ ]:
# What is the total number of models that grid search builds


Notice that like a normal estimator, this has not yet been applied to any data.
Calling the ``fit()`` method will fit the model at each grid point, keeping track of the scores along the way:

In [49]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19]),
                         'p': [1, 2], 'weights': ['uniform', 'distance']},
             scoring='precision_macro')

Now that this is fit, we can ask for the best parameters as follows:

In [ ]:
grid.best_params_

In [ ]:
grid

Finally, if we wish, we can use the best model and show the fit to our data using code from before:

In [ ]:
model = grid.best_estimator_
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
print('classification_report:\n',classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

The grid search provides many more options, including the ability to specify a custom scoring function, to parallelize the computations, to do randomized searches, and more.

In [ ]:
## Exercise:

Load the cancer dataset and choose the best classification algorithm with the best hyperparameters.

- Define X and y

- To simplify, remove missing values

- Split data to train and test

- Use 5 fold cross validation and grid search on train data

- Choose appropriate validation metric

- Set grid parameters for each classification algorithm

- Build the best models for each classification algorithm according to the best estimator (best hyperparameters) given by the grid search

- Compare the performance of the algorithms with the best hyperparametrs on the test data according to confusion matrix, recall, precision, F1, and auc metrics

In [32]:
df=pd.read_csv('../data/breast_cancer_wisconsin.csv')
df

,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2.0,1,1,1,0
695,841769,2,1,1,1,2,1.0,1,1,1,0
696,888820,5,10,10,3,7,3.0,8,10,2,1
697,897471,4,8,6,4,3,4.0,10,6,1,1


In [33]:
from sklearn.neighbors import KNeighborsClassifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC

In [36]:
df.head()


,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0


In [37]:
print(df.isna().sum())
df = df.dropna()

Id                  0
Cl.thickness        0
Cell.size           0
Cell.shape          0
Marg.adhesion       0
Epith.c.size        0
Bare.nuclei        16
Bl.cromatin         0
Normal.nucleoli     0
Mitoses             0
Class               0
dtype: int64


In [42]:
X = df.drop(columns=['Id','Class'], axis=1)
y = df['Class']
X.head()

,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses
0,5,1,1,1,2,1.0,3,1,1
1,5,4,4,5,7,10.0,3,2,1
2,3,1,1,1,2,2.0,3,1,1
3,6,8,8,1,3,4.0,3,7,1
4,4,1,1,3,2,1.0,3,1,1


In [43]:
df.Class.value_counts()

Class
0    444
1    239
Name: count, dtype: int64

In [44]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64

In [45]:
X.describe()

,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses
count,683.000000,683.000000,683.000000,683.000000,683.000000,683.000000,683.000000,683.000000,683.000000
mean,4.442167,3.150805,3.215227,2.830161,3.234261,3.544656,3.445095,2.869693,1.603221
std,2.820761,3.065145,2.988581,2.864562,2.223085,3.643857,2.449697,3.052666,1.732674
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,1.000000,1.000000,1.000000,2.000000,1.000000,2.000000,1.000000,1.000000
50%,4.000000,1.000000,1.000000,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000
75%,6.000000,5.000000,5.000000,4.000000,4.000000,6.000000,5.000000,4.000000,1.000000
max,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000


In [47]:
X_train, X_test, y_train, y_test=train_test_split(X,y,random_state=42, train_size=.80,stratify=y)

**Knn**

In [71]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Hyperparameters to test
param_grid = {
    'n_neighbors': range(1, 21),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]      # 1 = Manhattan, 2 = Euclidean
}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring='precision_macro',
    cv=5
)

# Train using training data
grid.fit(X_train, y_train)

# Get the best KNN model
best_knn = grid.best_estimator_

# Predict test data
y_pred_knn = best_knn.predict(X_test)

# Probability of the positive class for AUC
y_prob_knn = best_knn.predict_proba(X_test)[:, 1]

# Calculate performance metrics
knn_recall = recall_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn)
knn_auc = roc_auc_score(y_test, y_prob_knn)
knn_cm = confusion_matrix(y_test, y_pred_knn)

# Display best hyperparameters
print("Best KNN parameters:")
print(grid.best_params_)

print("\nBest cross-validation precision score:")
print(grid.best_score_)

# Display test results
print("\nConfusion Matrix:")
print(knn_cm)

print("KNN Recall   :", knn_recall)
print("KNN Precision:", knn_precision)
print("KNN F1 Score :", knn_f1)
print("KNN AUC      :", knn_auc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

Best KNN parameters:
{'n_neighbors': 12, 'p': 2, 'weights': 'distance'}

Best cross-validation precision score:
0.9763136375688471

Confusion Matrix:
[[85  4]
 [ 2 46]]
KNN Recall   : 0.9583333333333334
KNN Precision: 0.92
KNN F1 Score : 0.9387755102040817
KNN AUC      : 0.9822097378277154

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97        89
           1       0.92      0.96      0.94        48

    accuracy                           0.96       137
   macro avg       0.95      0.96      0.95       137
weighted avg       0.96      0.96      0.96       137



Exception ignored in: <function ResourceTracker.__del__ at 0x102399bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103f49bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1098cdbc0>
Traceback (most recent call last

**Decision Tree**

In [65]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Hyperparameters to test
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 2, 3, 4, 5, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Grid search with 5-fold cross-validation
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring='precision_macro',
    cv=5
)

grid.fit(X_train, y_train)
best_dt = grid.best_estimator_
y_pred_dt = best_dt.predict(X_test)
dt_recall = recall_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_cm = confusion_matrix(y_test, y_pred_dt)

print("Best Decision Tree parameters:")
print(grid.best_params_)
print("\nBest cross-validation score:")
print(grid.best_score_)
print("\nConfusion Matrix:")
print(dt_cm)

print("Decision Tree Recall   :", dt_recall)
print("Decision Tree Precision:", dt_precision)
print("Decision Tree F1 Score :", dt_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

Best Decision Tree parameters:
{'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}

Best cross-validation score:
0.9569549405766915

Confusion Matrix:
[[85  4]
 [ 3 45]]
Decision Tree Recall   : 0.9375
Decision Tree Precision: 0.9183673469387755
Decision Tree F1 Score : 0.9278350515463918

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.96        89
           1       0.92      0.94      0.93        48

    accuracy                           0.95       137
   macro avg       0.94      0.95      0.94       137
weighted avg       0.95      0.95      0.95       137



**Logistic Regression**

In [67]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    scoring='precision_macro',
    cv=5
)


grid.fit(X_train, y_train)

best_lr = grid.best_estimator_


y_pred_lr = best_lr.predict(X_test)

lr_recall = recall_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)

print("Best Logistic Regression parameters:")
print(grid.best_params_)
print("\nBest cross-validation precision score:")
print(grid.best_score_)
print("\nConfusion Matrix:")
print(lr_cm)

print("Logistic Regression Recall   :", lr_recall)
print("Logistic Regression Precision:", lr_precision)
print("Logistic Regression F1 Score :", lr_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-pa

Best Logistic Regression parameters:
{'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}

Best cross-validation precision score:
0.9705337945857835

Confusion Matrix:
[[85  4]
 [ 1 47]]
Logistic Regression Recall   : 0.9791666666666666
Logistic Regression Precision: 0.9215686274509803
Logistic Regression F1 Score : 0.9494949494949495

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97        89
           1       0.92      0.98      0.95        48

    accuracy                           0.96       137
   macro avg       0.95      0.97      0.96       137
weighted avg       0.96      0.96      0.96       137



**Support Vector Machine (SVM)**

In [68]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    scoring='precision_macro',
    cv=5
)

grid.fit(X_train, y_train)

best_svm = grid.best_estimator_

y_pred_svm = best_svm.predict(X_test)

y_prob_svm = best_svm.predict_proba(X_test)[:, 1]

svm_recall = recall_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
svm_auc = roc_auc_score(y_test, y_prob_svm)
svm_cm = confusion_matrix(y_test, y_pred_svm)

print("Best SVM parameters:")
print(grid.best_params_)

print("\nBest cross-validation precision score:")
print(grid.best_score_)
print("\nConfusion Matrix:")
print(svm_cm)

print("SVM Recall   :", svm_recall)
print("SVM Precision:", svm_precision)
print("SVM F1 Score :", svm_f1)
print("SVM AUC      :", svm_auc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-pa

Best SVM parameters:
{'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}

Best cross-validation precision score:
0.9697883916200556

Confusion Matrix:
[[85  4]
 [ 2 46]]
SVM Recall   : 0.9583333333333334
SVM Precision: 0.92
SVM F1 Score : 0.9387755102040817
SVM AUC      : 0.9927434456928839

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97        89
           1       0.92      0.96      0.94        48

    accuracy                           0.96       137
   macro avg       0.95      0.96      0.95       137
weighted avg       0.96      0.96      0.96       137



**PipeLine: Polynomial Logistic Regression**

In [69]:
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

poly_lr_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=5000))
])

param_grid = {
    'poly__degree': [1, 2],
    'poly__interaction_only': [False, True],
    'lr__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'lr__penalty': ['l1', 'l2'],
    'lr__solver': ['liblinear']
}

grid = GridSearchCV(
    estimator=poly_lr_pipeline,
    param_grid=param_grid,
    scoring='precision_macro',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_poly_lr = grid.best_estimator_
y_pred_poly_lr = best_poly_lr.predict(X_test)

y_prob_poly_lr = best_poly_lr.predict_proba(X_test)[:, 1]

poly_lr_recall = recall_score(y_test, y_pred_poly_lr)
poly_lr_precision = precision_score(y_test, y_pred_poly_lr)
poly_lr_f1 = f1_score(y_test, y_pred_poly_lr)
poly_lr_auc = roc_auc_score(y_test, y_prob_poly_lr)
poly_lr_cm = confusion_matrix(y_test, y_pred_poly_lr)

print("Best Polynomial Logistic Regression parameters:")
print(grid.best_params_)

print("\nBest cross-validation precision score:")
print(grid.best_score_)
print("\nConfusion Matrix:")
print(poly_lr_cm)

print(f"\nPolynomial LR Recall    : {poly_lr_recall:.4f}")
print(f"Polynomial LR Precision : {poly_lr_precision:.4f}")
print(f"Polynomial LR F1 Score  : {poly_lr_f1:.4f}")
print(f"Polynomial LR AUC       : {poly_lr_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_poly_lr,
        target_names=['Benign', 'Malignant']
    )
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-pa

Best Polynomial Logistic Regression parameters:
{'lr__C': 1, 'lr__penalty': 'l1', 'lr__solver': 'liblinear', 'poly__degree': 1, 'poly__interaction_only': False}

Best cross-validation precision score:
0.9721659967773707

Confusion Matrix:
[[85  4]
 [ 1 47]]

Polynomial LR Recall    : 0.9792
Polynomial LR Precision : 0.9216
Polynomial LR F1 Score  : 0.9495
Polynomial LR AUC       : 0.9909

Classification Report:
              precision    recall  f1-score   support

      Benign       0.99      0.96      0.97        89
   Malignant       0.92      0.98      0.95        48

    accuracy                           0.96       137
   macro avg       0.95      0.97      0.96       137
weighted avg       0.96      0.96      0.96       137



**Compare the best models on the test data**

In [72]:
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)

models = {
    "KNN": best_knn,
    "Decision Tree": best_dt,
    "Logistic Regression": best_lr,
    "SVM": best_svm,
    "Polynomial Logistic Regression": best_poly_lr
}

results = []

for name, model in models.items():

    # Predict labels
    y_pred = model.predict(X_test)

    # Predict probabilities for AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Recall": recall_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    })

comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

print(comparison)

                            Model    Recall  Precision  F1 Score       AUC
0             Logistic Regression  0.979167   0.921569  0.949495  0.990637
1  Polynomial Logistic Regression  0.979167   0.921569  0.949495  0.990871
2                             KNN  0.958333   0.920000  0.938776  0.982210
3                             SVM  0.958333   0.920000  0.938776  0.992743
4                   Decision Tree  0.937500   0.918367  0.927835  0.946278
